# Quantizing TabPFN v3: PTQ · Fine-tuned PTQ · QAT
## Dataset: OpenML Task 363629 (Diabetes)

Compares four variants on OpenML task 363629:

| # | Model | Description |
|---|---|---|
| 1 | **Vanilla** | TabPFN v3, float32, no fine-tuning |
| 2 | **PTQ** | Post-Training Quantization — dynamic int8 on vanilla weights |
| 3 | **Fine-tuned → PTQ** | Float32 fine-tuning (TabPFNFinetuner), then PTQ |
| 4 | **QAT** | Quantization-Aware Training — fake-quant nodes during fine-tuning, then convert to int8 |

**Quantization approaches:**
- **PTQ** (`quantize_dynamic`): converts `nn.Linear` weights to int8 after training. Fast, no data needed, ~4× size reduction.
- **QAT** (`prepare_qat` → fine-tune → `convert`): inserts differentiable fake-quant nodes so the model adapts its weights to the quantization grid. Typically recovers accuracy lost by PTQ.

In [ ]:
import sys
sys.path.insert(0, "..")

import copy
import io
import time
import warnings

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import torch
import torch.nn as nn
import openml
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from torch.optim import AdamW

warnings.filterwarnings("ignore")
print(f"PyTorch {torch.__version__}")

## 1. Load OpenML Task 363629

In [ ]:
TASK_ID = 363629

task    = openml.tasks.get_task(TASK_ID)
dataset = task.get_dataset()
X_df, y_series, _, _ = dataset.get_data(target=dataset.default_target_attribute)

# Encode categorical columns and target
X_df = X_df.apply(
    lambda col: LabelEncoder().fit_transform(col.astype(str)) if col.dtype == object else col
)
X = X_df.values.astype(np.float32)

# Impute NaNs with column means
col_means = np.nanmean(X, axis=0)
nan_mask  = np.isnan(X)
X[nan_mask] = np.take(col_means, np.where(nan_mask)[1])

le_target = LabelEncoder()
y = le_target.fit_transform(y_series.astype(str))

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train, y_train, test_size=0.2, random_state=42, stratify=y_train
)

print(f"Dataset : {dataset.name}")
print(f"Features: {X.shape[1]} | Classes: {len(le_target.classes_)} {list(le_target.classes_)}")
print(f"Train: {len(X_tr)} | Val: {len(X_val)} | Test: {len(X_test)}")
print(f"Class distribution (train): {dict(zip(*np.unique(y_tr, return_counts=True)))}")

## 2. Helper Functions

In [ ]:
def plot_decision_boundary_pca(clf, X, y, ax, title, resolution=60):
    """Decision boundary in PCA-2D space.
    Projects X to 2D, sweeps a grid, reconstructs to original space, queries clf.
    """
    pca  = PCA(n_components=2, random_state=42)
    X_2d = pca.fit_transform(X)

    x_min, x_max = X_2d[:, 0].min() - 0.5, X_2d[:, 0].max() + 0.5
    y_min, y_max = X_2d[:, 1].min() - 0.5, X_2d[:, 1].max() + 0.5
    xx, yy = np.meshgrid(
        np.linspace(x_min, x_max, resolution),
        np.linspace(y_min, y_max, resolution),
    )
    grid_orig = pca.inverse_transform(
        np.c_[xx.ravel(), yy.ravel()].astype(np.float32)
    ).astype(np.float32)

    proba     = clf.predict_proba(grid_orig)
    n_classes = proba.shape[1]
    if n_classes == 2:
        Z = proba[:, 1].reshape(xx.shape)
        ax.contourf(xx, yy, Z, levels=50, cmap=plt.cm.RdBu_r, alpha=0.75, vmin=0, vmax=1)
        ax.contour(xx, yy, Z, levels=[0.5], colors="k", linewidths=1.2)
    else:
        Z = proba.argmax(axis=1).reshape(xx.shape)
        ax.contourf(xx, yy, Z, levels=n_classes, alpha=0.5)

    ax.scatter(X_2d[:, 0], X_2d[:, 1], c=y,
               cmap="RdBu_r", edgecolors="w", linewidths=0.4, s=30, zorder=3, vmin=0, vmax=1)
    acc = (clf.predict(X) == y).mean()
    ax.set_title(f"{title}\nacc={acc:.3f}")
    ax.set_xlabel("PC1")
    ax.set_ylabel("PC2")


def model_size_mb(model):
    """Serialized state_dict size in MB (works for float32, fp16, and quantized models)."""
    buf = io.BytesIO()
    torch.save(model.state_dict(), buf)
    return buf.tell() / 1024 ** 2


def time_inference(clf, X, n_runs=5):
    """Average predict_proba latency in milliseconds over n_runs."""
    times = []
    for _ in range(n_runs):
        t0 = time.perf_counter()
        clf.predict_proba(X)
        times.append((time.perf_counter() - t0) * 1000)
    return float(np.mean(times))


class DirectModelClassifier:
    """Wraps a raw TabPFN nn.Module for sklearn-compatible predict/predict_proba.

    Runs the meta-learning forward pass: X_train as context, X_test as query.
    Compatible with float32, float16, and dynamic-quantized (int8/int4) models.

    Parameters
    ----------
    input_dtype : torch.dtype
        Dtype to cast input tensors to before passing to the model.
        Use torch.float16 for fp16 models, torch.float32 otherwise
        (quantized models expect float32 inputs even though weights are int8/int4).
    """

    def __init__(
        self,
        model: nn.Module,
        X_train: np.ndarray,
        y_train: np.ndarray,
        device: str = "cpu",
        input_dtype: torch.dtype = torch.float32,
    ):
        self.model       = model.eval()
        self.device      = device
        self.input_dtype = input_dtype
        self._le         = LabelEncoder().fit(y_train)
        self.X_train     = np.asarray(X_train, dtype=np.float32)
        self.y_enc       = self._le.transform(y_train)
        self.classes_    = self._le.classes_

    def predict_proba(self, X_test: np.ndarray) -> np.ndarray:
        X_test = np.asarray(X_test, dtype=np.float32)
        X_all  = np.concatenate([self.X_train, X_test], axis=0)
        x_t    = torch.tensor(X_all, dtype=self.input_dtype, device=self.device).unsqueeze(1)
        y_ctx  = torch.tensor(self.y_enc, dtype=torch.long, device=self.device).unsqueeze(1)
        with torch.no_grad():
            out = self.model(x_t, y_ctx, only_return_standard_out=True)
        # Cast back to float32 for softmax (fp16 output may overflow softmax)
        return torch.softmax(out[:, 0, :].float(), dim=-1).cpu().numpy()

    def predict(self, X_test: np.ndarray) -> np.ndarray:
        return self.classes_[self.predict_proba(X_test).argmax(axis=1)]


print("Helpers defined.")

## 3. Vanilla TabPFN v3 (Baseline)

Standard float32 inference — no quantization, no fine-tuning.

In [ ]:
from tabpfn import TabPFNClassifier

# Force CPU — quantized models only support CPU inference
device = "cpu"

clf_vanilla = TabPFNClassifier(n_estimators=8, device=device, random_state=42)
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    clf_vanilla.fit(X_tr, y_tr)

vanilla_raw  = clf_vanilla.models_[0]          # raw nn.Module — reused for quantization
acc_vanilla  = (clf_vanilla.predict(X_test) == y_test).mean()
size_vanilla = model_size_mb(vanilla_raw)
time_vanilla = time_inference(clf_vanilla, X_test)

print(f"Vanilla  | acc={acc_vanilla:.4f} | size={size_vanilla:.2f} MB | inference={time_vanilla:.1f} ms")

## 4. Post-Training Quantization (PTQ)

`torch.quantization.quantize_dynamic` converts all `nn.Linear` weight matrices from
float32 (4 bytes/param) to int8 (1 byte/param) at load time. Activations remain float32.
No calibration data needed — applied directly to the pretrained weights.

**Expected:** ~2–4× size reduction, minimal accuracy drop on most tabular tasks.

In [ ]:
ptq_model = copy.deepcopy(vanilla_raw).eval()
ptq_model = torch.quantization.quantize_dynamic(
    ptq_model, qconfig_spec={nn.Linear}, dtype=torch.qint8
)

clf_ptq  = DirectModelClassifier(ptq_model, X_tr, y_tr, device="cpu")
acc_ptq  = (clf_ptq.predict(X_test) == y_test).mean()
size_ptq = model_size_mb(ptq_model)
time_ptq = time_inference(clf_ptq, X_test)

print(f"PTQ      | acc={acc_ptq:.4f} | size={size_ptq:.2f} MB | inference={time_ptq:.1f} ms")
print(f"  → size reduction vs vanilla: {size_vanilla / size_ptq:.2f}×")
print(f"  → accuracy delta:            {(acc_ptq - acc_vanilla)*100:+.2f} pp")

## 5. Fine-tuned TabPFN v3 → then PTQ

Fine-tune in float32 first (using `TabPFNFinetuner`), then apply PTQ to the fine-tuned weights.
This tests whether fine-tuning improves accuracy both before and after quantization.

In [ ]:
from finetuning_engine import TabPFNFinetuner

clf_ft = TabPFNFinetuner(
    epochs=50,
    learning_rate=1e-4,
    query_ratio=0.3,
    grad_clip=1.0,
    patience=20,
    warmup_proportion=0.1,
    device=device,
    random_state=42,
    verbose=True,
)
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    clf_ft.fit(X_tr, y_tr, X_val, y_val)

In [ ]:
clf_ft.plot_history()

In [ ]:
# Apply PTQ to the fine-tuned float32 model
ft_raw = clf_ft._model

ft_ptq_model = copy.deepcopy(ft_raw).eval()
ft_ptq_model = torch.quantization.quantize_dynamic(
    ft_ptq_model, qconfig_spec={nn.Linear}, dtype=torch.qint8
)

# Fine-tuned float32 (reference)
acc_ft      = (clf_ft.predict(X_test) == y_test).mean()
size_ft     = model_size_mb(ft_raw)
time_ft     = time_inference(clf_ft, X_test)

# Fine-tuned + PTQ
clf_ft_ptq  = DirectModelClassifier(ft_ptq_model, X_tr, y_tr, device="cpu")
acc_ft_ptq  = (clf_ft_ptq.predict(X_test) == y_test).mean()
size_ft_ptq = model_size_mb(ft_ptq_model)
time_ft_ptq = time_inference(clf_ft_ptq, X_test)

print(f"Fine-tuned fp32  | acc={acc_ft:.4f} | size={size_ft:.2f} MB | inference={time_ft:.1f} ms")
print(f"Fine-tuned + PTQ | acc={acc_ft_ptq:.4f} | size={size_ft_ptq:.2f} MB | inference={time_ft_ptq:.1f} ms")
print(f"  → size reduction: {size_ft / size_ft_ptq:.2f}×  |  accuracy delta after PTQ: {(acc_ft_ptq - acc_ft)*100:+.2f} pp")

## 6. Quantization-Aware Training (QAT)

QAT is the principled approach to fine-tuning a model that will be quantized:

1. **Prepare** — `prepare_qat` inserts `FakeQuantize` nodes throughout the model.
   These are differentiable (straight-through estimator), so gradients flow normally.
2. **Fine-tune** — same meta-learning loop as `TabPFNFinetuner`.
   The model minimizes loss *under* quantization noise, pushing weights toward
   representable int8 values.
3. **Convert** — `torch.quantization.convert` replaces fake-quant nodes with
   true integer operations.

> If `prepare_qat` is incompatible with TabPFN's architecture the cell falls back
> gracefully to the Fine-tuned → PTQ model from step 5.

In [ ]:
# ── QAT helpers ───────────────────────────────────────────────────────────────

def _make_meta_batch(X, y, epoch, query_ratio=0.3, device="cpu", seed=42):
    rng     = np.random.default_rng(seed + epoch)
    n       = len(X)
    n_query = max(1, int(n * query_ratio))
    idx     = rng.permutation(n)
    ctx_idx, q_idx = idx[n_query:], idx[:n_query]
    X_all   = np.concatenate([X[ctx_idx], X[q_idx]], axis=0)
    x_t     = torch.tensor(X_all, dtype=torch.float32, device=device).unsqueeze(1)
    y_ctx   = torch.tensor(y[ctx_idx], dtype=torch.long, device=device).unsqueeze(1)
    y_q     = torch.tensor(y[q_idx],   dtype=torch.long, device=device)
    return x_t, y_ctx, y_q


def _validate_raw(model, X_tr, y_enc, X_val, y_val_enc, device="cpu"):
    X_all = np.concatenate([X_tr, X_val], axis=0)
    x_t   = torch.tensor(X_all,     dtype=torch.float32, device=device).unsqueeze(1)
    y_ctx = torch.tensor(y_enc,     dtype=torch.long,    device=device).unsqueeze(1)
    y_v   = torch.tensor(y_val_enc, dtype=torch.long)
    with torch.no_grad():
        out = model(x_t, y_ctx, only_return_standard_out=True)
    logits = out[:, 0, :]
    loss   = float(nn.functional.cross_entropy(logits, y_v).item())
    acc    = float((logits.argmax(1).cpu() == y_v).float().mean().item())
    return acc, loss


def _load_fresh_tabpfn(device="cpu"):
    clf = TabPFNClassifier(
        device=device,
        ignore_pretraining_limits=True,
        fit_mode="fit_preprocessors",
        inference_precision=torch.float32,
    )
    rng = np.random.default_rng(0)
    X_d = rng.standard_normal((10, 4)).astype(np.float32)
    y_d = np.array([0, 1, 0, 1, 0, 1, 0, 1, 0, 1])
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        clf.fit(X_d, y_d)
    return clf.models_[0]


print("QAT helpers ready.")

In [ ]:
# ── Prepare model for QAT ─────────────────────────────────────────────────────

le_qat    = LabelEncoder()
y_enc     = le_qat.fit_transform(y_tr)
y_val_enc = le_qat.transform(y_val)

# Detect quantization backend
backend = "fbgemm"
try:
    torch.backends.quantized.engine = "fbgemm"
except Exception:
    backend = "qnnpack"
    torch.backends.quantized.engine = "qnnpack"
print(f"Quantization backend: {backend}")

model_qat = copy.deepcopy(_load_fresh_tabpfn(device))
model_qat.train()

qat_supported = False
try:
    model_qat.qconfig = torch.quantization.get_default_qat_qconfig(backend)
    torch.quantization.prepare_qat(model_qat, inplace=True)
    qat_supported = True
    print("prepare_qat: OK — running QAT fine-tuning")
except Exception as e:
    print(f"prepare_qat failed: {e}")
    print("Fallback: will use Fine-tuned + PTQ model from step 5.")

In [ ]:
# ── QAT fine-tuning loop ──────────────────────────────────────────────────────

EPOCHS   = 50
LR       = 1e-4
PATIENCE = 20

if qat_supported:
    optimizer    = AdamW([p for p in model_qat.parameters() if p.requires_grad], lr=LR)
    best_acc     = -1.0
    best_state   = None
    patience_ctr = 0
    history_qat  = {"train_loss": [], "val_acc": []}

    for epoch in range(EPOCHS):
        model_qat.train()
        x_t, y_ctx_t, y_q_t = _make_meta_batch(X_tr, y_enc, epoch, device=device)

        optimizer.zero_grad(set_to_none=True)
        out  = model_qat(x_t, y_ctx_t, only_return_standard_out=True)
        loss = nn.functional.cross_entropy(out[:, 0, :], y_q_t)

        if torch.isfinite(loss):
            loss.backward()
            nn.utils.clip_grad_norm_(model_qat.parameters(), 1.0)
            optimizer.step()

        model_qat.eval()
        val_acc, val_loss = _validate_raw(model_qat, X_tr, y_enc, X_val, y_val_enc, device)
        history_qat["train_loss"].append(float(loss.detach()))
        history_qat["val_acc"].append(val_acc)

        print(f"  [QAT] epoch {epoch+1:3d}/{EPOCHS} | "
              f"train_loss={float(loss):.4f} | val_loss={val_loss:.4f} | val_acc={val_acc:.4f}")

        if val_acc > best_acc + 1e-4:
            best_acc     = val_acc
            best_state   = copy.deepcopy(model_qat.state_dict())
            patience_ctr = 0
        else:
            patience_ctr += 1
            if patience_ctr >= PATIENCE:
                print(f"  [QAT] early stopping at epoch {epoch + 1}")
                break

    model_qat.load_state_dict(best_state)
    model_qat.eval()
    print(f"  [QAT] best val_acc = {best_acc:.4f}")

    # Convert fake-quant → true int8
    try:
        torch.quantization.convert(model_qat, inplace=True)
        print("  [QAT] convert() → int8 model ready")
    except Exception as e:
        print(f"  [QAT] convert() failed ({e}); keeping float32 QAT model")

else:
    # Reuse Fine-tuned + PTQ model from step 5
    model_qat   = ft_ptq_model
    history_qat = None
    print("Using Fine-tuned + PTQ as QAT fallback.")

In [ ]:
# Plot QAT training history (only when QAT ran)
if history_qat is not None:
    ep = range(1, len(history_qat["train_loss"]) + 1)
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(ep, history_qat["train_loss"], marker="o", markersize=3)
    axes[0].set_title("QAT Train Loss"); axes[0].set_xlabel("Epoch"); axes[0].grid(True)
    axes[1].plot(ep, history_qat["val_acc"], color="green", marker="o", markersize=3)
    axes[1].set_title("QAT Validation Accuracy"); axes[1].set_xlabel("Epoch"); axes[1].grid(True)
    plt.tight_layout(); plt.show()

In [ ]:
clf_qat  = DirectModelClassifier(model_qat, X_tr, y_tr, device="cpu")
acc_qat  = (clf_qat.predict(X_test) == y_test).mean()
size_qat = model_size_mb(model_qat)
time_qat = time_inference(clf_qat, X_test)

print(f"QAT      | acc={acc_qat:.4f} | size={size_qat:.2f} MB | inference={time_qat:.1f} ms")
print(f"  → accuracy delta vs vanilla: {(acc_qat - acc_vanilla)*100:+.2f} pp")
print(f"  → accuracy delta vs PTQ:     {(acc_qat - acc_ptq)*100:+.2f} pp")

## 7. Comparison Summary

In [ ]:
results = [
    ("Vanilla (fp32)",        acc_vanilla,  size_vanilla,  time_vanilla),
    ("PTQ (int8)",            acc_ptq,      size_ptq,      time_ptq),
    ("Fine-tuned (fp32)",     acc_ft,       size_ft,       time_ft),
    ("Fine-tuned → PTQ",      acc_ft_ptq,   size_ft_ptq,   time_ft_ptq),
    ("QAT Fine-tuned (int8)", acc_qat,      size_qat,      time_qat),
]

hdr = f"{'Model':<26} {'Test Acc':>9} {'Size MB':>9} {'Inf ms':>8} {'Δ vs vanilla':>14}"
print(hdr)
print("-" * len(hdr))
for name, acc, sz, ms in results:
    delta = acc - acc_vanilla
    print(f"{name:<26} {acc:>9.4f} {sz:>9.2f} {ms:>8.1f} {delta:>+13.2%}")

In [ ]:
# Bar chart: accuracy, model size, inference latency
names     = [r[0] for r in results]
accs      = [r[1] for r in results]
sizes     = [r[2] for r in results]
inf_times = [r[3] for r in results]
colors    = ["#4C72B0", "#DD8452", "#55A868", "#C44E52", "#8172B2"]
x = np.arange(len(names))

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

for ax, vals, ylabel, title in zip(
    axes,
    [accs, sizes, inf_times],
    ["Test Accuracy", "Model Size (MB)", "Inference Time (ms)"],
    ["Test Accuracy", "Model Size", "Inference Latency"],
):
    ax.bar(x, vals, color=colors)
    ax.set_xticks(x)
    ax.set_xticklabels(names, rotation=20, ha="right", fontsize=9)
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    for i, v in enumerate(vals):
        ax.text(i, v * 1.01, f"{v:.3g}", ha="center", fontsize=8)

axes[0].set_ylim(0, 1.05)
fig.suptitle(f"OpenML Task {TASK_ID} — {dataset.name}", fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

## 8. Decision Boundary Comparison (PCA Projection)

Since the diabetes dataset has 8 features, we project test points to 2D with PCA,
sweep a grid, reconstruct back to 8D, and query each classifier.

In [ ]:
clf_vanilla_direct = DirectModelClassifier(copy.deepcopy(vanilla_raw).eval(), X_tr, y_tr)

clfs   = [clf_vanilla_direct, clf_ptq,   clf_ft,            clf_ft_ptq,        clf_qat]
titles = ["Vanilla (fp32)",   "PTQ int8", "Fine-tuned fp32", "Fine-tuned→PTQ",  "QAT int8"]

fig, axes = plt.subplots(1, 5, figsize=(22, 4.5))
for ax, clf, title in zip(axes, clfs, titles):
    plot_decision_boundary_pca(clf, X_test, y_test, ax, title, resolution=40)

if len(le_target.classes_) == 2:
    sm = plt.cm.ScalarMappable(cmap=plt.cm.RdBu_r, norm=mcolors.Normalize(vmin=0, vmax=1))
    fig.subplots_adjust(right=0.88)
    cbar_ax = fig.add_axes([0.90, 0.12, 0.015, 0.76])
    fig.colorbar(sm, cax=cbar_ax, label="P(class = 1)")

fig.suptitle(f"OpenML Task {TASK_ID} — {dataset.name} | PCA decision boundary",
             fontsize=12, y=1.02)
plt.show()